# Train a Streaming Wake Word — Kaggle / Colab

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/ww_train_streaming.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/ww_train_streaming.ipynb)

## Start here

This notebook trains a streaming wake-word model for the OVOS wake-word plugin. "Streaming" means the plugin works out the sound features of each new 80 ms of audio once and keeps them, instead of working them out again for the whole last 1.5 seconds; the model itself still looks at the last 1.5 seconds. It needs recordings of your phrase, which [`synthetic_ww_dataset.ipynb`](synthetic_ww_dataset.ipynb) makes. On Kaggle: run that notebook, open this one, click **Add Input**, pick the finished run under **Notebook output**, set `DATASET_DIRS = "/kaggle/input"` in the first code cell, turn Internet on and press **Run All**. It takes about 50 minutes and needs no GPU; at the end you get the model file, the lines for your OVOS configuration, and a few measurements of this one run.

On Colab: zip the dataset folder made by the first notebook, upload the zip with the file panel on the left, unzip it by running `!unzip -q run_1.zip -d /content/dataset` in a new code cell, and set `DATASET_DIRS = "/content/dataset"`.

| Field | Value |
|---|---|
| Inputs | one or more folders written by `synthetic_ww_dataset` (`DATASET_DIRS`) |
| Outputs | `ww_streaming_output/<phrase>_streaming.onnx` for `ovos-ww-plugin-wakeforge`, and `ovos_config.json` |
| Runtime | about 50 minutes; `SMOKE = True` a few minutes |
| Hardware | a CPU; a GPU only shortens the training step (about 20 minutes on a CPU) |
| Cost | about 50 minutes of a free Kaggle CPU session |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path. The smoke path speaks the phrase with `espeak-ng` and checks that the plugin loads the model as a streaming model.

Read more: [the plugin and its streaming models](https://github.com/OpenVoiceOS/ovos-ww-plugin-wakeforge), [the WakeHuBERT featurizer](https://huggingface.co/TigreGotico/wakehubert-tiny), [wakeforge](https://github.com/TigreGotico/wakeforge), whose `stream_heads.py` method this notebook follows.

In [ ]:
# Settings: the only cell a beginner changes.
SMOKE = False      # True: a tiny test run of a few minutes that needs no dataset; it uses its own minimums for the two numbers below
N_SAMPLES = 24     # SMOKE only: how many training clips of the phrase a robot voice makes (never fewer than 24)
EPOCHS = 10        # how many times the training goes over the data (SMOKE: never fewer than 20)
DATASET_DIRS = ""  # your dataset folder(s) from synthetic_ww_dataset, separated by commas; on Kaggle: "/kaggle/input"

## Advanced settings

Leave these alone unless you know why you change them.

In [ ]:
import os, time, json
from collections import Counter

WAKE_WORD   = os.environ.get("WAKE_WORD", "hey mycroft")   # SMOKE only; a real run takes the phrase from the dataset
WORK_DIR    = os.environ.get("WORK_DIR",   "/tmp/ww_streaming_work" if os.path.isdir("/kaggle") else "./ww_streaming_work")   # downloads
OUTPUT_DIR  = os.environ.get("OUTPUT_DIR", "./ww_streaming_output")   # the model file
SEED        = int(os.environ.get("SEED", "42"))                # fixes which clips are used and where they go, not the exact training result
DATASET_DIRS = os.environ.get("DATASET_DIRS", DATASET_DIRS)

TRAIN_SPEECH_SECONDS, TRAIN_SOUND_SECONDS = 16200, 10800   # training stream: 4.5 h of speech, 3 h of sound and music
STREAM_SECONDS = 7200                          # length of each of the calibration and test streams
SOUND_SHARE = 0.4                              # share of sound and music in those two streams
QUIET_SHARE = 0.04                             # near silence added to every stream
N_SPEECH_TRAIN, N_SPEECH_STREAM = 800, 1200    # single spoken words: training stream, and the two other streams together
SEGMENT_SECONDS = 60                           # the training stream is processed in pieces of this length
VALIDATION_SHARE = 0.1                         # last part of the training stream, with clips kept out of training
GAP_SECONDS = (1.5, 5.0)                       # pause between two phrases mixed into the training stream
SNR_DB = (0.0, 25.0)                           # how much louder than the background a mixed-in phrase is
POS_FROM, POS_TO, IGNORE_TO = 2, 14, 30        # frames after a phrase ends: "wake word" from 2 to 14, not used until 30
LR, BATCH_SIZE = 1.5e-3, 256
NEG_PER_POS, NEG_POOL = 5, 100000              # background blocks per phrase block; pool the hardest ones are picked from
CAL_MAX_FA_PER_HOUR = 2.0                      # threshold 0.5 is set to give at most this many false wake-ups per hour on the calibration stream
TYPICAL_PHRASE_SCORE = 0.9                     # the score a typical set-aside phrase gets once the scores are adjusted
CONTEXT_SNR_DB = (5.0, 10.0, 20.0)             # loudness of the test phrases above their background

SMOKE_MIN_EPOCHS = 20
if not SMOKE and EPOCHS < 1:
    raise ValueError(f"EPOCHS is {EPOCHS}: the training needs at least 1 round. Set EPOCHS to 1 or more in the first code cell (10 is the usual value).")
if SMOKE:
    N_SAMPLES = max(N_SAMPLES, 24)
    N_SPEECH_TRAIN, N_SPEECH_STREAM = 60, 40
    TRAIN_SPEECH_SECONDS, TRAIN_SOUND_SECONDS, STREAM_SECONDS = 240, 120, 90
    SEGMENT_SECONDS, VALIDATION_SHARE, GAP_SECONDS, NEG_POOL = 20, 0.15, (1.0, 2.5), 20000
    EPOCHS = max(EPOCHS, SMOKE_MIN_EPOCHS)
    LR, BATCH_SIZE = float(os.environ.get("SMOKE_LR", "3e-3")), 64

## Step 1 — Find your dataset

The notebook looks for your dataset folders and checks that they fit together. Without a dataset it stops here and tells you what to do.

In [ ]:
# Hidden helper: finds the folders named in DATASET_DIRS and refuses the ones that cannot be trained on together.
_STEPS, _last = {}, [time.time()]
def tick(name):
    # remembers how long each step took
    now = time.time()
    _STEPS[name] = now - _last[0]
    _last[0] = now

if isinstance(DATASET_DIRS, str):
    DATASET_DIRS = [p.strip() for p in DATASET_DIRS.replace("\n", ",").split(",") if p.strip()]
MSWC_TEST_PERCENT = 20                          # share of the copied voices on the test side in synthetic_ww_dataset
DONOR_SPLIT = f"MLCommons/ml_spoken_words speaker_id, crc32 % 100 < {MSWC_TEST_PERCENT} is test"
DATASETS = []                                   # (folder, its dataset.json)
for root in DATASET_DIRS:
    root = os.path.abspath(os.path.expanduser(root))
    assert os.path.isdir(root), f"DATASET_DIRS names {root}, which is not a folder."
    found, seen, visited = [], [], set()
    for d, subdirs, files in os.walk(root, followlinks=True):   # Kaggle mounts attached inputs a few levels down, sometimes as links
        real = os.path.realpath(d)
        if real in visited:                     # a link to a folder already searched, or a link loop
            subdirs[:] = []
            continue
        visited.add(real)
        seen.append(d)
        if "dataset.json" in files and "metadata.csv" in files:
            found.append(real)
            subdirs[:] = []                     # a dataset's own audio folders are not searched
        elif d[len(root):].count(os.sep) >= 6:
            subdirs[:] = []
    assert found, (f"No folder written by synthetic_ww_dataset (one holding dataset.json and metadata.csv) was found under {root}. "
                   f"Folders searched: {', '.join(seen[:12])}{' ...' if len(seen) > 12 else ''}")
    for d in sorted(found):
        if d not in (x for x, _ in DATASETS):
            DATASETS.append((d, json.load(open(os.path.join(d, "dataset.json"), encoding="utf-8"))))
if DATASETS:
    _norm = lambda s: " ".join(str(s).lower().split())
    _list = "; ".join(f"{d}: {m.get('wake_word')!r} ({m.get('language')}), run {m.get('run_id')}" for d, m in DATASETS)
    if any(m.get("format") != "ovos-synthetic-ww-dataset" for _, m in DATASETS):
        raise ValueError(f"A folder in DATASET_DIRS was not written by synthetic_ww_dataset: {_list}")
    if len({_norm(m["wake_word"]) for _, m in DATASETS}) > 1 or len({m["language"] for _, m in DATASETS}) > 1:
        raise ValueError(f"The folders in DATASET_DIRS are for different phrases; give it folders made for one phrase. Found: {_list}")
    _twice = [r for r, n in Counter(m["run_id"] for _, m in DATASETS).items() if n > 1]
    if _twice:
        raise ValueError(f"Several folders were made with RUN_ID {_twice}, so their voices are the same voices. "
                         f"Generate each folder with its own RUN_ID. Found: {_list}")
    _old = [d for d, m in DATASETS if m.get("donor_split") != DONOR_SPLIT]
    if _old:
        raise ValueError(f"{', '.join(_old)} was made by an older synthetic_ww_dataset that split its copied voices by another rule, so its test "
                         "voices could be heard in training here. Make it again with the current synthetic_ww_dataset.")
    if not SMOKE and any(m.get("smoke") for _, m in DATASETS):
        raise ValueError("A folder in DATASET_DIRS was made with SMOKE = True (a robot voice, for testing only); it is not training data.")
    WAKE_WORD = DATASETS[0][1]["wake_word"]
    print(f"Found {len(DATASETS)} dataset folder(s) for the phrase {WAKE_WORD!r}:")
    for d, m in DATASETS:
        print(f"   {d}: run {m['run_id']}, {m['counts']['positives']['train']} train and {m['counts']['positives']['test']} test clips")
elif not SMOKE:
    raise ValueError("DATASET_DIRS is empty. This notebook trains on a dataset made by synthetic_ww_dataset: run that notebook first, "
                     "then name its output folder in DATASET_DIRS in the first code cell (Kaggle: attach the finished run with Add Input, "
                     "Notebook output, and set DATASET_DIRS = \"/kaggle/input\"). Or set SMOKE = True for the small test run.")
else:
    print(f"SMOKE without a dataset: a robot voice will say {WAKE_WORD!r}.")
WORD_SLUG = WAKE_WORD.strip().lower().replace(" ", "_").replace("-", "_")

## Step 2 — Install

This installs the OVOS wake-word plugin and a few audio packages. If it fails with a network error, Internet is off (Kaggle: Settings panel, Internet).

In [ ]:
import sys, shutil

%pip install --quiet --pre "ovos-ww-plugin-wakeforge>=0.18.0a1" "onnx>=1.22.0" "onnxruntime>=1.30.0" "numpy>=2.0" "soundfile>=0.14.0" "scipy>=1.11" "huggingface_hub>=0.30" "pyarrow>=15"

try:
    import torch                               # Kaggle and Colab already have it; it is not installed here
except ImportError as exc:
    raise ImportError("torch is not installed. Kaggle and Colab have it; elsewhere run: pip install torch") from exc
from importlib.metadata import version

DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        (torch.ones(8, device="cuda") * 2).sum().item()
        DEVICE = "cuda"
    except Exception as exc:
        print(f"A GPU is listed but does not work ({exc}); training on the CPU.")
THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(THREADS)
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | plugin ovos-ww-plugin-wakeforge {version('ovos-ww-plugin-wakeforge')}")
print(f"The model trains on: {DEVICE}. Everything else runs on the CPU ({THREADS} threads).")
if SMOKE and not DATASETS:
    assert shutil.which("espeak-ng"), "SMOKE speaks the phrase with espeak-ng, which is not installed (Linux: apt-get install espeak-ng)."
os.makedirs(WORK_DIR, exist_ok=True)
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
assert free_gb > (2 if SMOKE else 8), f"Only {free_gb:.1f} GB of disk is free; this run needs about {2 if SMOKE else 8} GB."
tick("install")

## Step 3 — Load the streaming listener

WakeHuBERT turns sound into numbers the model can learn from. Here it runs as a streaming graph: it gets 80 ms of sound at a time and remembers what it heard, exactly as the plugin will run it on your device.

In [ ]:
import math, random
from pathlib import Path
import numpy as np
import onnxruntime as ort
import soundfile as sf
from scipy.signal import fftconvolve, resample_poly
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

FEATURIZER_NAME = "wakehubert-int8"
FEATURIZER_REPO = "TigreGotico/wakehubert-tiny"
FEATURIZER_REVISION = "343b26ea8881b5a01447a1659555941b25220d22"
SR, HOP, BLOCK = 16000, 320, 1280          # samples per second, per feature frame, per 80 ms block
FPB = BLOCK // HOP                         # 4 feature frames per block
WINDOW_FRAMES = 75                         # the model looks at the last 75 frames: 1.5 s
WINDOW = WINDOW_FRAMES * HOP

ort.set_default_logger_severity(3)          # no warnings about the model files' internals
opts = ort.SessionOptions()
opts.intra_op_num_threads, opts.inter_op_num_threads = THREADS, 1
stream_sess = ort.InferenceSession(hf_hub_download(FEATURIZER_REPO, "wakehubert_int8_stream.onnx", revision=FEATURIZER_REVISION),
                                   opts, providers=["CPUExecutionProvider"])
_in_names = [i.name for i in stream_sess.get_inputs()]
_state_shapes = [i.shape for i in stream_sess.get_inputs()[1:]]

def to_int16(a):
    return (np.clip(a, -1, 1) * 32767).astype(np.int16)

def stream_features(audio):
    # Feeds audio the way the plugin does: memory at zero, 1.5 s of silence first, then 80 ms blocks with the memory carried over.
    # Returns [75 + frames, 128]; the first 75 frames are that starting silence.
    assert len(audio) % BLOCK == 0
    audio = to_int16(audio).astype(np.float32) / 32768.0       # a microphone gives the plugin 16-bit samples
    state = [np.zeros(s, np.float32) for s in _state_shapes]
    feats, *state = stream_sess.run(None, dict(zip(_in_names, [np.zeros((1, WINDOW), np.float32), *state])))
    out = [feats[0]]
    for i in range(0, len(audio), BLOCK):
        feats, *state = stream_sess.run(None, dict(zip(_in_names, [audio[None, i:i + BLOCK], *state])))
        out.append(feats[0])
    return np.concatenate(out)

probe = stream_features(np.zeros(2 * BLOCK, np.float32))
FEATURE_DIM = probe.shape[1]
assert probe.shape == (WINDOW_FRAMES + 2 * FPB, 128)
print(f"Streaming featurizer ready: every 80 ms block gives {FPB} frames of {FEATURE_DIM} numbers; it carries {len(_state_shapes)} pieces of memory.")

## Step 4 — Download background sound

The model must learn what is *not* the wake word: ordinary speech, sounds and music from public datasets. They are split three ways, for training, for setting the threshold and for the final test. No recording or clip is in two of the three.

In [ ]:
# Hidden helper: downloads spoken words, sounds, music, room echoes and LibriSpeech sentences, and splits them three ways.
import io, re, tarfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
import pyarrow.parquet as pq

rng = random.Random(SEED)
api = HfApi()
AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg")
COLLECTION = "TigreGotico/notwakeword-datasets"
SPEECH_REPO = "TigreGotico/not-wake-words-speech-en"
RIR_REPO = "davidscripka/MIT_environmental_impulse_responses"
# Sound and music sets, read as a few Parquet shards each; the number is the most clips taken from a set.
SOUND_SETS = ({"TigreGotico/public_domain_sounds_3secs": 8, "TigreGotico/FMA_3secs": 8, "TigreGotico/NAR": 8, "TigreGotico/ambient_noises": 8} if SMOKE else
              {"TigreGotico/public_domain_sounds_3secs": 2500, "TigreGotico/ambient_noises": 1500, "TigreGotico/ESC-50": 1500,
               "TigreGotico/FMA_3secs": 503, "TigreGotico/building_106_kitchen_3secs": 800, "TigreGotico/NAR": 600})
AUDIOSET = "agkphysics/AudioSet"                       # 10 s clips, a few of its 700 MB shards
AUDIOSET_TRAIN_SHARDS, AUDIOSET_CAL_SHARD, AUDIOSET_TEST_SHARD, AUDIOSET_PER_SHARD = [3, 17, 31], 4, 20, 600

def to16k(x, sr):
    x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(SR, sr)
        x = resample_poly(x, SR // g, sr // g).astype(np.float32)
    return x

def load16k(path):
    return to16k(*sf.read(path, dtype="float32", always_2d=True))

def trim(x, db=-40.0):
    # cuts the silence off both ends
    peak = np.abs(x).max()
    if peak == 0:
        return x
    idx = np.nonzero(np.abs(x) > peak * 10 ** (db / 20))[0]
    return x[idx[0]: idx[-1] + 1]

def load_set(repo, n, folder):
    files = sorted(f for f in api.list_repo_files(repo, repo_type="dataset") if f.lower().endswith(AUDIO_EXT))
    names = [f for f in rng.sample(files, min(n, len(files))) if not any(c in f for c in "[]*?")]
    snapshot_download(repo, repo_type="dataset", allow_patterns=names, local_dir=Path(WORK_DIR) / folder, max_workers=8)
    clips = []                                        # [(file name, audio)]
    for name in names:
        try:
            clips.append((Path(name).name, load16k(Path(WORK_DIR) / folder / name)))
        except Exception as exc:                      # an unreadable file is skipped
            print(f"skipped {name}: {exc}")
    return clips

def parquet_clips(repo, filename, cap, tag, revision=None):
    # up to `cap` clips spread evenly over one Parquet shard
    pf = pq.ParquetFile(hf_hub_download(repo, filename, repo_type="dataset", revision=revision))
    col = next(f.name for f in pf.schema_arrow if "bytes" in str(f.type))
    stride, out, k = max(1, pf.metadata.num_rows // cap), [], 0
    for batch in pf.iter_batches(batch_size=64, columns=[col]):
        for item in batch.column(0).to_pylist():
            if k % stride == 0 and len(out) < cap and item.get("bytes"):
                try:
                    x = to16k(*sf.read(io.BytesIO(item["bytes"]), dtype="float32", always_2d=True))
                except Exception:
                    x = None
                if x is not None and len(x) > 0.1 * SR and np.abs(x).max() > 1e-4:
                    out.append((f"{tag}/{Path(item.get('path') or '').name or filename}#{k}", x))
            k += 1
    return out

def source_of(name):
    # The recording a sound clip was cut from. Three sets hold several clips of one recording: ESC-50 names its
    # recording in the second part of the file name, and the two "3secs" sets end the file name in a three-digit counter.
    tag, file = name.split("#")[0].split("/", 1)
    if tag == "ESC-50":
        return f"{tag}/{file.split('-')[1]}"
    if tag in ("public_domain_sounds_3secs", "building_106_kitchen_3secs"):
        return f"{tag}/{re.sub(r'[0-9]{3}[.]wav$', '', file)}"
    return f"{tag}/{file}"

def split3(clips, r):
    # training stream / calibration stream / test stream, by source recording: all clips of one recording go the same way
    groups = {}
    for clip in clips:
        groups.setdefault(source_of(clip[0]), []).append(clip)
    order = sorted(groups)
    r.shuffle(order)
    want, parts = max(1, round(0.13 * len(clips))), ([], [], [])     # (calibration, test, training)
    for src in order:                                               # a recording too large for a held-out part goes to training
        k = next((k for k in (0, 1) if len(parts[k]) < want and len(parts[k]) + len(groups[src]) <= 2 * want), 2)
        parts[k].extend(groups[src])
    return parts[2], parts[0], parts[1]

t_dl = time.time()
speech = [(n, trim(x)) for n, x in load_set(SPEECH_REPO, N_SPEECH_TRAIN + N_SPEECH_STREAM, "speech")]
speech = [(n, x) for n, x in speech if len(x) > 0.1 * SR]
speech_train, _rest = speech[:N_SPEECH_TRAIN], speech[N_SPEECH_TRAIN:]
speech_cal, speech_tst = _rest[:len(_rest) // 2], _rest[len(_rest) // 2:]

sounds = {"train": [], "cal": [], "tst": []}
for repo, cap in SOUND_SETS.items():
    shards = sorted(f for f in api.list_repo_files(repo, repo_type="dataset", revision="refs/convert/parquet") if f.endswith(".parquet"))
    clips = []
    for fn in shards:
        clips += parquet_clips(repo, fn, -(-cap // len(shards)), repo.split("/")[1], revision="refs/convert/parquet")
    for key, part in zip(sounds, split3(clips[:cap], random.Random(f"{SEED}:{repo}"))):
        sounds[key] += part
if not SMOKE:
    files = sorted(f for f in api.list_repo_files(AUDIOSET, repo_type="dataset") if f.endswith(".parquet"))
    bal, ev = [f for f in files if f.startswith("data/bal_train/")], [f for f in files if f.startswith("data/eval/")]
    for i in AUDIOSET_TRAIN_SHARDS:
        sounds["train"] += parquet_clips(AUDIOSET, bal[i], AUDIOSET_PER_SHARD, "AudioSet")
    sounds["cal"] += parquet_clips(AUDIOSET, ev[AUDIOSET_CAL_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
    sounds["tst"] += parquet_clips(AUDIOSET, ev[AUDIOSET_TEST_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
_src = {key: {source_of(n) for n, _ in sounds[key]} for key in sounds}
assert not (_src["train"] & _src["cal"] or _src["train"] & _src["tst"] or _src["cal"] & _src["tst"]), "a sound recording is in two of the three streams"
assert not ({n for n, _ in speech_train} & {n for n, _ in speech_cal + speech_tst} or {n for n, _ in speech_cal} & {n for n, _ in speech_tst}), "a spoken word is in two of the three streams"
rirs = [r[:int(0.6 * SR)] for _, r in load_set(RIR_REPO, 8 if SMOKE else 60, "rir") if len(r) > 100]
print(f"Spoken words: {len(speech_train)} for training, {len(speech_cal)} for the calibration stream, {len(speech_tst)} for the test stream")
print(f"Sound clips : {len(sounds['train'])} for training, {len(sounds['cal'])} for the calibration stream, {len(sounds['tst'])} for the test stream; "
      f"{len(rirs)} room echoes ({time.time() - t_dl:.0f} s)")
for key in sounds:
    print(f"   {key}: " + ", ".join(f"{k} {v}" for k, v in sorted(Counter(n.split("/")[0] for n, _ in sounds[key]).items())) + f" ({len(_src[key])} recordings)")
print("   No sound recording is in two of the three.")

libri = {}                                             # LibriSpeech: read sentences, the real run only (about 700 MB)
cal_speakers = tst_speakers = []
if not SMOKE:
    def get_libri(name):
        tgz = Path(WORK_DIR) / f"{name}.tar.gz"
        if not tgz.exists():
            urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{name}.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(Path(WORK_DIR) / "librispeech", filter="data")
        tgz.unlink()
        out = {}                                      # speaker -> [(flac path, transcript)]
        for tr in sorted((Path(WORK_DIR) / "librispeech" / "LibriSpeech" / name).glob("*/*/*.trans.txt")):
            for line in tr.read_text().splitlines():
                uid, text = line.split(" ", 1)
                out.setdefault(tr.parent.parent.name, []).append((tr.parent / f"{uid}.flac", text.lower()))
        return out
    with ThreadPoolExecutor(2) as ex:
        libri = dict(zip(("dev-clean", "test-clean"), ex.map(get_libri, ("dev-clean", "test-clean"))))
    tc = sorted(libri["test-clean"])
    random.Random(SEED).shuffle(tc)
    cal_speakers, tst_speakers = tc[:20], tc[20:]       # the calibration stream and the test stream share no speaker
    print(f"LibriSpeech : {len(libri['dev-clean'])} speakers for training, {len(cal_speakers)} for the calibration stream, {len(tst_speakers)} others for the test stream")
tick("downloads")

## Step 5 — Load your recordings of the phrase

The clips of your phrase are read from the dataset, each on the side (training or test) the dataset gave it. A tenth of the training clips are set aside to pick the best training round later.

In [ ]:
import csv, subprocess, tempfile, zlib

pos = {"train": [], "test": []}        # lists of (voice id, audio)

def keep(x, sr=SR):
    # trims the silence at both ends; drops clips longer than 1.5 s, which the model cannot see whole
    x = trim(to16k(np.asarray(x, np.float32).reshape(len(x), -1), sr), db=-35.0)
    return x if 0.2 * SR <= len(x) <= WINDOW else None

skipped = Counter()
for d, m in DATASETS:
    with open(Path(d) / "metadata.csv", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["label"] != "1" or row["source"].endswith("+augmented"):   # only the phrase, without the dataset's noisy copies
                continue
            assert row["split"] in pos, f"{d}/metadata.csv: unknown split {row['split']!r}"
            x = keep(*sf.read(Path(d) / row["path"], dtype="float32"))
            if x is None:
                skipped["longer than 1.5 s or too short"] += 1
            else:
                pos[row["split"]].append((row["voice_id"], x))
if not DATASETS:                                       # SMOKE: a robot voice says the phrase
    variants = [f"en-{a}+{g}{n}" for a in ("us", "gb") for g in "mf" for n in range(1, 5)]
    rng.shuffle(variants)
    for split, n, pool in (("train", N_SAMPLES, variants[4:]), ("test", 8, variants[:4])):
        for i in range(n):
            with tempfile.TemporaryDirectory() as d:
                subprocess.run(["espeak-ng", "-v", pool[i % len(pool)], "-p", str(rng.randint(20, 80)), "-s", str(rng.randint(120, 190)),
                                "-w", f"{d}/a.wav", WAKE_WORD], check=True, capture_output=True)
                x = keep(*sf.read(f"{d}/a.wav", dtype="float32"))
            if x is not None:
                pos[split].append((f"espeak:{pool[i % len(pool)]}", x))

voices = {s: {v for v, _ in pos[s]} for s in pos}
assert not voices["train"] & voices["test"], "a voice is on both sides of the split"
_need = 8 if SMOKE else 400
if len(pos["train"]) < _need or len(pos["test"]) < 4:
    raise ValueError(f"Too few clips of the phrase: {len(pos['train'])} for training and {len(pos['test'])} for the test, but at least {_need} and 4 are needed. "
                     "This notebook expects a full run of synthetic_ww_dataset, which makes about 2,000 clips; a SMOKE run of it makes too few.")
val_voices = {v for v in voices["train"] if zlib.crc32(v.encode()) % 10 == 0} or {sorted(voices["train"])[0]}
pos_fit = [(v, x) for v, x in pos["train"] if v not in val_voices]
pos_val = [(v, x) for v, x in pos["train"] if v in val_voices]
is_copied = lambda voice: voice.startswith("mswc:")   # a voice copied from a real speaker; the others are invented from a description
print(f"Phrase {WAKE_WORD!r}: {len(pos['train'])} training clips and {len(pos['test'])} test clips"
      + ("; skipped " + ", ".join(f"{n} {k}" for k, n in skipped.items()) if skipped else ""))
print(f"   test clips in voices copied from real speakers: {sum(is_copied(v) for v, _ in pos['test'])} (these speakers are not in the training clips)")
print(f"   test clips in invented voices: {sum(not is_copied(v) for v, _ in pos['test'])} (new clips, but made from the same kind of voice descriptions as the training clips)")
print(f"Set aside to pick the best training round: {len(pos_val)} training clips")
POS_ORIGIN = (f"loaded from {len(DATASETS)} folder(s) written by synthetic_ww_dataset (runs " + ", ".join(m["run_id"] for _, m in DATASETS)
              + "), made with " + "; ".join(sorted({str(m.get("generator", "an unnamed generator")) for _, m in DATASETS}))
              if DATASETS else "generated with espeak-ng (SMOKE)")
tick("load the phrase")

## Step 6 — Build three long recordings

The background sound is glued into three continuous streams: about eight hours for training, two hours for choosing the threshold and two hours for the final test. Your phrase is then mixed into the training stream every few seconds, at different loudness and sometimes with room echo, and the notebook notes where each one ends.

In [ ]:
# Hidden helper: glues clips into streams and mixes the phrase into a stream at known places.
def peak_to(x, level):
    p = np.abs(x).max()
    return x * (level / p) if p > 0 else x

duration = lambda parts: sum(len(x) for *_, x in parts) / SR

def sentence_parts(speakers, split, seconds, seed):
    utts = [(spk, u) for spk in speakers for u in libri[split][spk]]
    random.Random(seed).shuffle(utts)
    parts, total = [], 0.0
    for spk, (path, text) in utts:
        if total >= seconds:
            break
        x = load16k(path)
        total += len(x) / SR
        parts.append(("LibriSpeech sentence", f"{spk}/{path.stem}: {text}", x))
    return parts

def quiet_parts(seconds, seed):
    # stretches of 2 to 6 s of digital silence or of very soft noise (-60 or -40 dB)
    r, parts, total = np.random.default_rng(seed), [], 0.0
    while total < seconds:
        n, level = int(r.uniform(2, 6) * SR), [None, -60.0, -40.0][len(parts) % 3]
        x = np.zeros(n, np.float32) if level is None else (r.standard_normal(n) * 10 ** (level / 20)).astype(np.float32)
        parts.append(("near silence", f"near silence {len(parts)}", x))
        total += n / SR
    return parts

def fill(parts, seconds, what):
    # takes parts up to `seconds`; repeats them when the sources are too short (the SMOKE run)
    have = duration(parts)
    if parts and have < seconds:
        print(f"   {what}: only {have / 60:.1f} of {seconds / 60:.1f} min of sources; they are repeated")
        parts = parts * math.ceil(seconds / have)
    out, total = [], 0.0
    for p in parts:
        if total >= seconds:
            break
        out.append(p); total += len(p[2]) / SR
    return out

def build_stream(words, sound_named, speakers, split, speech_seconds, sound_seconds, seed, what):
    # returns the audio and an index [(start s, end s, kind, name)] used to trace a false wake-up to its source
    r = np.random.default_rng(seed)
    sound_named = [sound_named[i] for i in r.permutation(len(sound_named))]
    parts = [("speech word", n, x) for n, x in words]
    parts = fill(parts, speech_seconds, f"{what}, spoken words") if not speakers else parts
    parts += fill([("sound: " + n.split("/")[0], n, x) for n, x in sound_named], sound_seconds, f"{what}, sounds")
    if speakers:
        parts += sentence_parts(speakers, split, speech_seconds - duration(parts[:len(words)]), seed + 2)
    parts += quiet_parts(QUIET_SHARE * (speech_seconds + sound_seconds), seed + 3)
    gap = np.zeros(int(0.3 * SR), np.float32)
    audio, index, t = [], [], 0.0
    for i in r.permutation(len(parts)):
        src, name, x = parts[i]
        if src != "near silence":                      # every clip gets a random loudness
            x = peak_to(x, float(r.uniform(0.05, 0.8) if src.startswith("sound") else r.uniform(0.2, 0.8)))
        index.append((t, t + len(x) / SR, src, name))
        audio += [x, gap]
        t += len(x) / SR + 0.3
    audio = np.concatenate(audio).astype(np.float32)
    return audio[:len(audio) // BLOCK * BLOCK], index

def describe(name, audio, index):
    kinds = Counter()
    for t0_, t1_, src, _ in index:
        kinds[src.split(":")[0]] += t1_ - t0_
    print(f"{name:11s} stream: {len(audio) / SR / 3600:.2f} h (" + ", ".join(f"{k} {v / 60:.0f} min" for k, v in kinds.items()) + ")")

def reverb_or_speed(x, r):
    if r.random() < 0.5:                               # a little faster or slower
        x = np.interp(np.arange(0, len(x), r.uniform(0.9, 1.1)), np.arange(len(x)), x).astype(np.float32)
    if rirs and r.random() < 0.5:                      # room echo
        rms = np.sqrt((x ** 2).mean())
        x = fftconvolve(x, rirs[int(r.integers(len(rirs)))])[:len(x) + int(0.1 * SR)]
        x = (x * (rms / (np.sqrt((x ** 2).mean()) + 1e-8))).astype(np.float32)
    return x

def place(audio, clips, r, start, stop, piece, change=True, gap=GAP_SECONDS, snr=None):
    # Walks from `start` to `stop` and mixes one clip into `audio` after every gap. Returns [(start sample, end sample, voice)].
    # A clip never crosses the border between two pieces of `piece` samples.
    spans, order, k = [], [], 0
    t = start + int(r.uniform(1.0, 3.0) * SR)
    while True:
        if k == len(order):
            order, k = list(r.permutation(len(clips))), 0
        voice, x = clips[order[k]]
        x = reverb_or_speed(peak_to(x, 0.7), r) if change else peak_to(x, 0.7)
        if (t + len(x) + SR) // piece != t // piece:
            t = (t // piece + 1) * piece + int(r.uniform(1.0, 3.0) * SR)
        if t + len(x) + SR > stop:
            return spans
        e = t + len(x)
        under = np.sqrt((audio[max(0, t - SR):e] ** 2).mean())            # loudness of the background around the clip
        gain = under * 10 ** ((r.uniform(*SNR_DB) if snr is None else float(r.choice(snr))) / 20) / (np.sqrt((x ** 2).mean()) + 1e-8)
        if under < 1e-3 or (snr is None and r.random() < 0.15):           # over near silence, or now and then: a loudness of its own
            gain = r.uniform(0.05, 0.7) / np.abs(x).max()
        audio[t:e] = np.clip(audio[t:e] + min(gain, 0.9 / np.abs(x).max()) * x, -1, 1)
        spans.append((t, e, voice))
        t = e + int(r.uniform(*gap) * SR)
        k += 1

In [ ]:
train_audio, train_index = build_stream(speech_train, sounds["train"], sorted(libri.get("dev-clean", {})), "dev-clean",
                                        TRAIN_SPEECH_SECONDS, TRAIN_SOUND_SECONDS, SEED + 10, "training stream")
cal_audio, cal_index = build_stream(speech_cal, sounds["cal"], cal_speakers, "test-clean",
                                    (1 - SOUND_SHARE) * STREAM_SECONDS, SOUND_SHARE * STREAM_SECONDS, SEED + 20, "calibration stream")
tst_audio, tst_index = build_stream(speech_tst, sounds["tst"], tst_speakers, "test-clean",
                                    (1 - SOUND_SHARE) * STREAM_SECONDS, SOUND_SHARE * STREAM_SECONDS, SEED + 30, "test stream")
SEGMENT = SEGMENT_SECONDS * SR
N_SEGMENTS = len(train_audio) // SEGMENT
N_VAL_SEGMENTS = max(1, round(VALIDATION_SHARE * N_SEGMENTS))
assert N_SEGMENTS >= 2, "the training stream is too short"
train_audio = train_audio[:N_SEGMENTS * SEGMENT]                 # whole pieces only
FIT_END = (N_SEGMENTS - N_VAL_SEGMENTS) * SEGMENT                # where the part with the set-aside clips starts
CAL_HOURS, STREAM_HOURS, TRAIN_HOURS = len(cal_audio) / SR / 3600, len(tst_audio) / SR / 3600, len(train_audio) / SR / 3600
for name, a, ix in (("training", train_audio, train_index), ("calibration", cal_audio, cal_index), ("test", tst_audio, tst_index)):
    describe(name, a, ix)

pr = np.random.default_rng(SEED + 40)
fit_spans = place(train_audio, pos_fit, pr, 0, FIT_END, SEGMENT)                       # the phrase goes into the training stream only
val_spans = place(train_audio, pos_val, pr, FIT_END, len(train_audio), SEGMENT)        # set-aside clips, in its last part
assert fit_spans and val_spans, "the training stream is too short to hold the phrase"
print(f"The phrase was mixed into the training stream {len(fit_spans)} times ({len(fit_spans) / len(pos_fit):.1f} times per clip), "
      f"and {len(val_spans)} times with the set-aside clips in its last {N_VAL_SEGMENTS * SEGMENT_SECONDS / 60:.0f} minutes.")
tick("build the streams")

## Step 7 — Listen to the training stream

The whole training stream goes through the streaming listener of Step 3, 80 ms at a time. Each 80 ms block then gets a label: "wake word" if the phrase ended a moment ago (40 to 280 ms), "not wake word" otherwise.

In [ ]:
t0 = time.time()
SEG_FRAMES = WINDOW_FRAMES + SEGMENT // HOP                       # frames kept per piece: the starting silence, then the audio
feats = np.empty((N_SEGMENTS, SEG_FRAMES, FEATURE_DIM), np.float16)   # kept as 16-bit floats to save memory
for k in range(N_SEGMENTS):                                       # every piece starts the way the plugin starts listening
    feats[k] = stream_features(train_audio[k * SEGMENT:(k + 1) * SEGMENT])
    if k and k % 120 == 0:
        print(f"  listened to {k * SEGMENT_SECONDS / 3600:.1f} of {TRAIN_HOURS:.1f} h ({time.time() - t0:.0f} s)")
FEATURE_SECONDS = time.time() - t0
print(f"Listened to {TRAIN_HOURS:.2f} h in {FEATURE_SECONDS:.0f} s, block by block ({TRAIN_HOURS * 3600 / FEATURE_SECONDS:.0f} times faster than real time)")

# a label for every frame of the stream: 1 wake word, 0 not, -1 not used (the phrase is half heard or sliding away)
labels = np.zeros(len(train_audio) // HOP, np.int8)
for s, e, _ in fit_spans + val_spans:
    sf_, ef = s // HOP, e // HOP
    labels[(sf_ + ef) // 2: ef + IGNORE_TO + 1] = -1
    labels[ef + POS_FROM: ef + POS_TO + 1] = 1

# one row per 80 ms block: where its 75 frames start in the features, its label, and the sample it ends at
seg_len = SEGMENT // HOP
ends = np.arange(FPB, seg_len + 1, FPB)
row_start = (np.arange(N_SEGMENTS)[:, None] * SEG_FRAMES + ends[None, :]).reshape(-1)
row_label = labels.reshape(N_SEGMENTS, seg_len)[:, ends - 1].reshape(-1)
row_time = (np.arange(N_SEGMENTS)[:, None] * seg_len + ends[None, :]).reshape(-1) * HOP
row_is_val = row_time > FIT_END
fit_pos, fit_neg = np.flatnonzero((row_label == 1) & ~row_is_val), np.flatnonzero((row_label == 0) & ~row_is_val)
val_pos, val_neg = np.flatnonzero((row_label == 1) & row_is_val), np.flatnonzero((row_label == 0) & row_is_val)
print(f"Blocks to train on: {len(fit_pos)} wake word, {len(fit_neg)} not wake word. Set aside: {len(val_pos)} and {len(val_neg)}.")
assert len(fit_pos) and len(val_pos)
tick("listen to the training stream")

## Step 8 — Train the model

A small network learns to tell the two kinds of block apart. After every round it is shown again the background it most mistakes for the phrase. The notebook keeps the round that finds most of the set-aside clips while waking at most once by mistake on their part of the stream.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    # reads the last 75 frames and gives one score
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(FEATURE_DIM, 128, batch_first=True)
        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 1)
    def forward(self, features):                 # [batch, frames, 128] -> [batch]
        out, _ = self.gru(features)
        return self.fc2(F.relu(self.fc1(out.mean(1)))).squeeze(-1)

feats_t = torch.from_numpy(feats.reshape(-1, FEATURE_DIM)).to(DEVICE)
_offsets = torch.arange(WINDOW_FRAMES, device=DEVICE)

def gather(rows):
    # the 75 frames each of the given blocks looks at
    return feats_t[torch.from_numpy(row_start[rows]).to(DEVICE)[:, None] + _offsets].float()

def scores_of(head, rows, bs=2048):
    head.eval()
    with torch.no_grad():
        return np.concatenate([head(gather(rows[i:i + bs])).cpu().numpy() for i in range(0, len(rows), bs)])

def wake_ups(scores, threshold):
    # the blocks where the plugin would wake: a score at or above the threshold, then 2 s (25 blocks) in which it ignores scores
    above, out, j = np.flatnonzero(scores >= threshold), [], 0
    while j < len(above):
        out.append(int(above[j]))
        j = int(np.searchsorted(above, above[j] + 26))
    return out

def lowest_threshold(scores, allowed):
    # the lowest threshold that lets at most `allowed` wake-ups through, found by halving the range 40 times
    lo, hi = float(scores[np.isfinite(scores)].min()), float(scores.max()) + 1e-3
    for _ in range(40):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if len(wake_ups(scores, mid)) > allowed else (lo, mid)
    return hi

val_rows = np.flatnonzero(row_is_val)                  # every block of the set-aside part, in time order
def set_aside(head):
    # the best score of each set-aside phrase among its "wake word" blocks, and the scores of the set-aside background blocks
    scores, t = scores_of(head, val_rows), row_time[val_rows]
    phrases = np.array([scores[np.searchsorted(t, e + POS_FROM * HOP):np.searchsorted(t, e + (POS_TO + 1) * HOP + 1)].max(initial=-np.inf)
                        for _, e, _ in val_spans])
    return phrases, np.where(row_label[val_rows] == 0, scores, -np.inf)

torch.manual_seed(SEED)
tr = np.random.default_rng(SEED + 50)
head = Head().to(DEVICE)
opt = torch.optim.Adam(head.parameters(), lr=LR)
pool = fit_neg[tr.choice(len(fit_neg), min(len(fit_neg), NEG_POOL), replace=False)]
n_neg = min(NEG_PER_POS * len(fit_pos), len(pool))
hard = pool[tr.choice(len(pool), n_neg // 2, replace=False)]      # before the first round the "hard" half is random
best, t0 = (-1.0, None, 0), time.time()
for epoch in range(1, EPOCHS + 1):
    t1 = time.time()
    rows = np.concatenate([fit_pos, hard, pool[tr.choice(len(pool), n_neg - len(hard), replace=False)]])
    y = torch.zeros(len(rows), device=DEVICE)
    y[:len(fit_pos)] = 1
    order = tr.permutation(len(rows))
    head.train()
    for i in range(0, len(order), BATCH_SIZE):
        b = order[i:i + BATCH_SIZE]
        loss = F.binary_cross_entropy_with_logits(head(gather(rows[b])), y[torch.from_numpy(b).to(DEVICE)])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    hard = pool[np.argsort(scores_of(head, pool))[len(pool) - n_neg // 2:]]            # the background it likes most after this round
    phrases, background = set_aside(head)
    found = float((phrases > lowest_threshold(background, 1)).mean())                   # at a threshold that wakes at most once on the set-aside background
    if found > best[0]:
        best = (found, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, epoch)
    print(f"round {epoch:2d}/{EPOCHS}: finds {found:.1%} of the {len(val_spans)} set-aside phrases ({time.time() - t1:.0f} s)")
head.load_state_dict(best[1])
TRAIN_SECONDS = time.time() - t0
print(f"Training took {TRAIN_SECONDS:.0f} s on {DEVICE}. Kept round {best[2]}.")
tick("train")

## Step 9 — Save the model and load it in the OVOS plugin

The model is saved with a note inside that says "trained on streamed features", which is how the plugin knows to run it as a streaming model. The plugin then loads it, and the cell checks that the plugin gives the same scores as the training did.

In [ ]:
# Hidden helper: writes the model file with its notes, and drives the plugin block by block.
import warnings
import onnx
from scipy.stats import chi2
from ovos_ww_plugin_wakeforge import WakeForgeHotwordPlugin, model_names

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
HEAD_PATH = Path(OUTPUT_DIR) / f"{WORD_SLUG}_streaming.onnx"
OVOS_BASE = {"module": "ovos-ww-plugin-wakeforge", "model": str(HEAD_PATH), "onnx_threads": THREADS}

class Adjusted(nn.Module):
    # the trained network, with its score scaled and shifted inside the saved file
    def __init__(self, net, scale, shift):
        super().__init__()
        self.net, self.scale, self.shift = net, scale, shift
    def forward(self, features):
        return self.scale * self.net(features) + self.shift

def export_head(scale=1.0, shift=0.0, threshold=None):
    head_cpu = Head()
    head_cpu.load_state_dict({k: v.cpu() for k, v in head.state_dict().items()})
    head_cpu = Adjusted(head_cpu, float(scale), float(shift)).eval()
    example = torch.from_numpy(feats[:8, 100:100 + WINDOW_FRAMES].astype(np.float32))
    kw = dict(input_names=["features"], output_names=["logits"],
              dynamic_axes={"features": {0: "batch", 1: "frames"}, "logits": {0: "batch"}}, opset_version=17)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            torch.onnx.export(head_cpu, example, str(HEAD_PATH), dynamo=False, **kw)
        except TypeError:                              # older torch has no dynamo argument
            torch.onnx.export(head_cpu, example, str(HEAD_PATH), **kw)
    model = onnx.load(str(HEAD_PATH))
    meta = {                                           # the notes the plugin reads
        "pretrained_featurizer": FEATURIZER_NAME,      # which featurizer to download
        "featurizer_revision": FEATURIZER_REVISION,
        "featurizer_features": "streamed",             # run the featurizer as a streaming graph
        "window_frames": WINDOW_FRAMES,                # score the last 75 frames after every block
        "feature_dim": FEATURE_DIM, "frame_rate_hz": 50.0, "arch": "gru", "wake_word": WAKE_WORD, "license": "Apache-2.0",
        "training_data": f"positives {POS_ORIGIN}, mixed into a {TRAIN_HOURS:.1f} h stream of {COLLECTION}{' and LibriSpeech dev-clean' if libri else ''}; "
                         f"room responses from {RIR_REPO}; streamed {FEATURIZER_NAME} features",
    }
    if threshold is not None:
        meta["default_threshold"] = repr(float(threshold))
        meta["score_scale"], meta["score_shift"] = repr(float(scale)), repr(float(shift))
        meta["threshold_note"] = (f"scores adjusted so that 0.5 gives at most {CAL_MAX_FA_PER_HOUR} false wake-ups per hour on a held-out "
                                  f"stream of {CAL_HOURS:.2f} h; a short stream, not the stream calibration of wakeforge")
    for key, value in meta.items():
        entry = model.metadata_props.add()
        entry.key, entry.value = key, str(value)
    onnx.save(model, str(HEAD_PATH))
    with torch.no_grad():                              # the saved file must give the same scores as torch
        want = head_cpu(example).numpy()
    got = ort.InferenceSession(str(HEAD_PATH), opts, providers=["CPUExecutionProvider"]).run(None, {"features": example.numpy()})[0]
    assert np.abs(got.reshape(-1) - want).max() < 1e-3, "the saved model and torch disagree"

def load_plugin(config):
    # the plugin as OVOS builds it; the second value is True when it runs the model on streamed features
    pl = WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config=dict(config))
    return pl, getattr(pl.engine, "stream", None) is not None

def run_stream(config, audio, label=None, fire=True):
    # Feeds the plugin like the OVOS listener: 16-bit audio in 80 ms blocks, asking found_wake_word() after each.
    # Returns the wake-up times (s), the score of every block, the plugin, and whether it used streamed features.
    # fire=False sets a threshold no score reaches, to record the raw scores of an unbroken stream.
    pcm = to_int16(audio)
    pl, streamed = load_plugin(config if fire else {**config, "threshold": 2.0})
    fires, scores, t1 = [], [], time.time()
    for i in range(0, len(pcm) - BLOCK + 1, BLOCK):
        pl.update(pcm[i:i + BLOCK].tobytes())
        scores.append(pl.last_score or 0.0)
        if pl.found_wake_word():
            fires.append((i + BLOCK) / SR)
        if label and (i // BLOCK) % 9000 == 0 and i:
            print(f"  {label}: {i / SR / 60:.0f} min done ({time.time() - t1:.0f} s)")
    return fires, np.array(scores), pl, streamed

def rate_interval(k, hours):
    # The range the true rate per hour plausibly lies in, given k events in `hours` hours (exact 95 % Poisson interval).
    # Few events give a wide range: the count is a rough measurement.
    return (0.0 if k == 0 else chi2.ppf(0.025, 2 * k) / 2 / hours), chi2.ppf(0.975, 2 * k + 2) / 2 / hours

def by_source(fires, index, top=6):
    # which kind of sound, and which clips, the wake-ups fell on
    starts, where = np.array([t0_ for t0_, *_ in index]), Counter()
    for t_ in fires:
        k = max(0, int(np.searchsorted(starts, t_, side="right")) - 1)
        where[(index[k][2], index[k][3])] += 1
    kinds = Counter()
    for (src, _), n in where.items():
        kinds[src] += n
    if kinds:
        print("   on: " + ", ".join(f"{k} {n}" for k, n in kinds.most_common()))
    for (src, name), n in where.most_common(top):
        print(f"   {n}x  {name[:100]}")

def found_clips(fires, spans):
    # per clip: did a wake-up fall between its start and 1.5 s after its end; and how many wake-ups fell outside every clip
    fires = np.array(fires)
    inside = [(fires >= s / SR) & (fires <= e / SR + 1.5) for s, e, _ in spans]
    return np.array([m.any() for m in inside]), int((~np.any(inside, axis=0)).sum()) if len(fires) else 0

In [ ]:
export_head()                                          # first as trained; Step 10 sets the threshold
plugin, STREAMED = load_plugin(OVOS_BASE)
print(f"Wrote {HEAD_PATH} ({HEAD_PATH.stat().st_size / 1024:.0f} KB)")
print(f"Plugin loaded the model: featurizer {plugin.featurizer.name}, streamed features {STREAMED}, looks at {plugin.engine.window} frames every {plugin.block} samples")
assert STREAMED, "the plugin did not load the model as a streamed-features model"
assert plugin.engine.window == WINDOW_FRAMES and plugin.block == BLOCK

# Same sound, two routes: the plugin on 16-bit audio, and the model on features made the way Step 7 made them. The scores must agree.
piece = train_audio[(N_SEGMENTS - N_VAL_SEGMENTS) * SEGMENT:][:SEGMENT]
_, from_plugin, _, _ = run_stream(OVOS_BASE, piece, fire=False)
with torch.no_grad():
    looked_at = torch.from_numpy(stream_features(piece)).to(DEVICE)[torch.from_numpy(ends).to(DEVICE)[:, None] + _offsets]
    from_training = torch.sigmoid(head(looked_at)).cpu().numpy()
PLUGIN_GAP = float(np.abs(from_plugin - from_training).max())
print(f"Plugin against training on {len(from_plugin)} blocks: largest difference in score {PLUGIN_GAP:.1e}")
assert PLUGIN_GAP < 1e-3, "the plugin does not give the model the features it was trained on"

## Step 10 — Set the threshold

The plugin listens to the two-hour calibration stream, which has no wake word in it, so every wake-up is a false one. The notebook then adjusts the model's scores so that the usual threshold, 0.5, gives at most 2 false wake-ups per hour on that stream, saves the model, and lets the plugin listen again to check the count. Later you can raise the threshold for fewer false wake-ups, or lower it to be heard more easily.

In [ ]:
_, cal_scores, _, _ = run_stream(OVOS_BASE, cal_audio, "calibration stream", fire=False)
cal_raw = np.log(np.clip(cal_scores, 1e-15, 1 - 1e-15) / (1 - np.clip(cal_scores, 1e-15, 1 - 1e-15)))   # back to the network's own scale
phrases, _ = set_aside(head)
typical = float(np.median(phrases))                    # the score of a typical set-aside phrase, on the same scale
ALLOWED = int(CAL_MAX_FA_PER_HOUR * CAL_HOURS)         # false wake-ups allowed on the calibration stream
THRESHOLD = 0.5
for allowed in range(ALLOWED, -1, -1):
    edge = lowest_threshold(cal_raw, allowed)          # just above the score that would add one more wake-up
    above = cal_raw[cal_raw >= edge]
    below = float(cal_raw[cal_raw < edge].max())
    raw_threshold = (below + float(above.min())) / 2 if len(above) else below + 0.5   # halfway to the next score up, so rounding cannot move a score across it
    SCALE = math.log(TYPICAL_PHRASE_SCORE / (1 - TYPICAL_PHRASE_SCORE)) / (typical - raw_threshold) if typical > raw_threshold else 1.0
    SHIFT = -SCALE * raw_threshold                     # after this adjustment the threshold found above is exactly 0.5
    adjust = lambda raw: 1 / (1 + np.exp(-(SCALE * raw + SHIFT)))
    export_head(SCALE, SHIFT, THRESHOLD)               # the adjustment and the threshold are stored in the model file
    OVOS_HOTWORD = {**OVOS_BASE, "threshold": THRESHOLD}
    # the check: the saved model, run by the plugin as on a device (after a wake-up the plugin starts listening afresh, which can change later scores)
    cal_fires, _, _, _ = run_stream(OVOS_HOTWORD, cal_audio, "calibration stream, saved model")
    CAL_N = len(cal_fires)
    print(f"Saved model, run by the plugin on the calibration stream: {CAL_N} false wake-ups in {CAL_HOURS:.2f} h (allowed: {ALLOWED}; "
          f"predicted from the recorded scores: {len(wake_ups(adjust(cal_raw), THRESHOLD))})")
    if CAL_N <= ALLOWED:
        break
    if allowed:
        print("   More than allowed: the threshold is moved up one step and the check is repeated.")
assert CAL_N <= ALLOWED, f"the saved model wakes {CAL_N} times on the calibration stream; at most {ALLOWED} are allowed"
if typical <= raw_threshold:
    print("A typical set-aside phrase scores below the threshold: expect the model to miss most phrases.")
print(f"\n{'threshold':>10} {'false wake-ups on the calibration stream':>42} {'per hour':>9} {'set-aside phrases found':>25}")
for cand in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    n = len(wake_ups(adjust(cal_raw), cand))
    print(f"{cand:>10} {n:>42} {n / CAL_HOURS:>9.2f} {(adjust(phrases) >= cand).mean():>25.1%}" + ("   <- used" if cand == THRESHOLD else ""))
print("(this table is predicted from the recorded scores; the line above it is the plugin's own count)")
by_source(cal_fires, cal_index)

cfg = {"listener": {"wake_word": WORD_SLUG},
       "hotwords": {WORD_SLUG: {"module": "ovos-ww-plugin-wakeforge",
                                "model": f"~/.local/share/mycroft/{HEAD_PATH.name}",   # where you copy the file to
                                "threshold": THRESHOLD}}}
(Path(OUTPUT_DIR) / "ovos_config.json").write_text(json.dumps(cfg, indent=2) + "\n")
tick("set the threshold")

## Step 11 — Test: false wake-ups

The plugin listens to the two-hour test stream, which nothing above has used and which has no wake word in it. Every wake-up is a mistake; the cell counts them and shows which sounds caused them.

In [ ]:
t0 = time.time()
fires, _, plugin_s, STREAMED = run_stream(OVOS_HOTWORD, tst_audio, "test stream")
print(f"Plugin loaded the model for the test: streamed features {STREAMED}, threshold {plugin_s.threshold}, pause after a wake-up {plugin_s.debounce_sec} s")
assert STREAMED and plugin_s.threshold == THRESHOLD
FA_PER_HOUR = len(fires) / STREAM_HOURS
FA_LO, FA_HI = rate_interval(len(fires), STREAM_HOURS)
print(f"Test stream: {len(fires)} false wake-ups in {STREAM_HOURS:.2f} h = {FA_PER_HOUR:.2f} per hour")
print(f"   With so few events the count is rough: the true rate is plausibly anywhere from {FA_LO:.2f} to {FA_HI:.2f} per hour (95 % interval).")
by_source(fires, tst_index)
tick("test: false wake-ups")

## Step 12 — Test: does it hear the phrase?

The test clips of the phrase are mixed into the start of the test stream, one every few seconds and 5, 10 or 20 dB louder than it; a clip counts as heard when the plugin wakes up between its start and 1.5 seconds after its end. Only the voices copied from real speakers are truly new to the model: an invented voice is a new clip, but it comes from the same kind of voice description as the training clips.

In [ ]:
cr = np.random.default_rng(SEED + 60)
need = sum(len(x) for _, x in pos["test"]) + 8 * SR * len(pos["test"])
ctx_audio = np.tile(tst_audio, math.ceil(need / len(tst_audio)))[:need // BLOCK * BLOCK].copy()
ctx_spans = place(ctx_audio, pos["test"], cr, 0, len(ctx_audio), piece=1 << 60, change=False, gap=(4.0, 7.0), snr=CONTEXT_SNR_DB)[:len(pos["test"])]
ctx_audio = ctx_audio[:(ctx_spans[-1][1] + 3 * SR) // BLOCK * BLOCK]
CTX_HOURS = len(ctx_audio) / SR / 3600

ctx_fires, _, _, _ = run_stream(OVOS_HOTWORD, ctx_audio, "phrase test")
ctx_hit, CTX_OUTSIDE = found_clips(ctx_fires, ctx_spans)
copied = np.array([is_copied(v) for _, _, v in ctx_spans])
CTX_N, CTX_DET = len(ctx_spans), int(ctx_hit.sum())
COPIED_N, COPIED_DET, INVENTED_N, INVENTED_DET = int(copied.sum()), int(ctx_hit[copied].sum()), int((~copied).sum()), int(ctx_hit[~copied].sum())
share = lambda k, n: f"{k} of {n}" + (f" = {k / n:.1%}" if n else "")
print("Phrase test:")
print(f"   voices copied from real speakers who are not in the training clips: {share(COPIED_DET, COPIED_N)} clips detected")
print(f"   invented voices (new clips, same kind of voice descriptions as in training): {share(INVENTED_DET, INVENTED_N)} clips detected")
print(f"   all test clips: {share(CTX_DET, CTX_N)} detected; {CTX_OUTSIDE} wake-ups outside any clip in {CTX_HOURS:.2f} h")
tick("test: the phrase")

## Step 13 — For reference: the plugin's ready-made model

If the plugin ships a ready-made streaming model for your phrase, it takes the same two tests on the same sound, at its own threshold. This only shows that the tests give sensible counts. It is not a ranking: the test clips come from the same generator this notebook's model was trained on, which favours this model, and the ready-made model's threshold was set by another rule.

In [ ]:
REFERENCE, REF = f"wakehubert_{WORD_SLUG}_streaming", None
if SMOKE:
    print("SMOKE: skipped.")
elif REFERENCE not in model_names():
    print(f"The plugin has no ready-made model named {REFERENCE}; nothing to run.")
else:
    ref_cfg = {"module": "ovos-ww-plugin-wakeforge", "model": REFERENCE, "onnx_threads": THREADS}
    ref_fires, _, ref_plugin, ref_streamed = run_stream(ref_cfg, tst_audio, f"{REFERENCE}, test stream")
    ref_hit, ref_outside = found_clips(run_stream(ref_cfg, ctx_audio)[0], ctx_spans)
    REF = (REFERENCE, ref_plugin.threshold, len(ref_fires), *rate_interval(len(ref_fires), STREAM_HOURS), int(ref_hit.sum()), ref_outside)
    print(f"{REFERENCE} (streamed features {ref_streamed}, its own threshold {REF[1]}):")
    print(f"   test stream: {REF[2]} false wake-ups in {STREAM_HOURS:.2f} h = {REF[2] / STREAM_HOURS:.2f} per hour (95 % interval {REF[3]:.2f} to {REF[4]:.2f})")
    print(f"   phrase test: copied voices {share(int(ref_hit[copied].sum()), COPIED_N)}, invented voices {share(int(ref_hit[~copied].sum()), INVENTED_N)}, "
          f"all {share(REF[5], CTX_N)} detected; {REF[6]} wake-ups outside any clip")
    print("   Not a ranking: the test clips come from the same generator this notebook's model was trained on, which favours this model.")
tick("reference model")

## What you got

- **A model file**, `ww_streaming_output/<phrase>_streaming.onnx` (Kaggle: Output tab; Colab: file panel, download it before the session ends). Copy it to the folder `~/.local/share/mycroft/` on your OVOS device.
- **The OVOS settings** printed below, also saved as `ovos_config.json`: add them to `~/.config/mycroft/mycroft.conf`. Install the plugin with `pip install --pre "ovos-ww-plugin-wakeforge>=0.18.0a1"`; it downloads the streaming featurizer by itself.
- **Measurements of this one run.** They come from synthetic voices and two hours of public speech, sound and music, and most test voices are invented the same way as the training voices. They do not say how the model does with real people in a real home. The threshold was set on only two hours of sound, and nothing was tested against phrases that sound like yours.

One measured run: on a free Kaggle CPU session, with the dataset of one full `synthetic_ww_dataset` run for "hey mycroft" and `SEED = 20261014`, the notebook took 49 minutes. The plugin woke 12 times by mistake in 2.22 hours of test stream, 5.40 per hour (plausibly 2.79 to 9.44), against the 2 per hour the threshold was set for on the calibration stream. It detected 52 of 68 test clips in voices copied from real speakers and 193 of 229 in invented voices. On the same audio the plugin's ready-made `wakehubert_hey_mycroft_streaming` woke 0 times and detected 53 of 68 copied-voice clips: it wakes by mistake less, and finds about the same share of truly new voices. What this notebook adds is a model for a phrase of your own. The settings were chosen while looking at these test sources, so the figures are optimistic.

In [ ]:
print("=" * 64)
print(f"Wake word {WAKE_WORD!r}, threshold {THRESHOLD} (the plugin woke {CAL_N} times in {CAL_HOURS:.2f} h of calibration stream; allowed {ALLOWED})")
print(f"Trained on    : {len(pos['train'])} clips, {POS_ORIGIN}; mixed {len(fit_spans)} times into {TRAIN_HOURS:.2f} h of stream; kept round {best[2]} of {EPOCHS}")
print(f"Plugin        : ovos-ww-plugin-wakeforge {version('ovos-ww-plugin-wakeforge')} ran the model on streamed features: {STREAMED}")
print(f"False wake-ups: {len(fires)} in {STREAM_HOURS:.2f} h of test stream = {FA_PER_HOUR:.2f} per hour (plausibly {FA_LO:.2f} to {FA_HI:.2f}, 95 % interval)")
print(f"Phrase heard  : copied real-speaker voices {share(COPIED_DET, COPIED_N)}; invented voices {share(INVENTED_DET, INVENTED_N)}; all {share(CTX_DET, CTX_N)}; "
      f"{CTX_OUTSIDE} wake-ups outside any clip in {CTX_HOURS:.2f} h")
if REF:
    print(f"Reference     : {REF[0]} at its own threshold {REF[1]}: {REF[2]} false wake-ups on the test stream, {REF[5]} of {CTX_N} clips detected "
          "(not a ranking: the test clips come from this model's own generator)")
print("Run time      : " + ", ".join(f"{k} {v / 60:.1f} min" for k, v in _STEPS.items()) + f"; total {sum(_STEPS.values()) / 60:.0f} min; model trained on {DEVICE}")
print("These numbers describe this one run. No claim is made about performance in a real home.")
print("=" * 64)
print("\nAdd this to mycroft.conf:\n" + json.dumps(cfg, indent=2))

if SMOKE:   # the test of the notebook itself: checks on the SAVED file and the settings a user copies
    to_raw = lambda p: np.log(np.clip(p, 1e-15, 1 - 1e-15) / (1 - np.clip(p, 1e-15, 1 - 1e-15)))
    # 1. the plugin runs the saved file the way this notebook assumed: streamed features, wake on one score, 2 s pause
    assert STREAMED, "the plugin did not run the saved model on streamed features"
    mode = (plugin_s.smoother.method, plugin_s.smoother.window_size, plugin_s.smoother.patience, plugin_s.debounce_sec)
    print(f"\nSMOKE check: plugin smoothing {mode[0]} over {mode[1]} block(s), patience {mode[2]}, pause {mode[3]} s (must be max, 1, 1, 2.0)")
    assert mode == ("max", 1, 1, 2.0), "the plugin does not run the saved model in the mode its threshold was set for"
    # 2. the configuration a user copies names the threshold that was set
    saved = json.loads((Path(OUTPUT_DIR) / "ovos_config.json").read_text())["hotwords"][WORD_SLUG]["threshold"]
    assert saved == THRESHOLD, f"ovos_config.json names threshold {saved}, the notebook set {THRESHOLD}"
    # 3. the saved file gives exactly the trained network's scores with the adjustment (Step 9 recorded the network's own scores on this audio)
    _, smoke_scores, _, _ = run_stream(OVOS_HOTWORD, train_audio[FIT_END:], fire=False)
    gap = float(np.abs(smoke_scores[:len(from_plugin)] - adjust(to_raw(from_plugin))).max())
    print(f"SMOKE check: saved file against the trained network with its adjustment, largest difference {gap:.1e} (must be below 1e-4)")
    assert gap < 1e-4, "the saved file does not give the trained network's adjusted scores"
    # 4. the rule Step 10 uses to predict wake-ups from recorded scores matches the plugin, on audio where it wakes. They can differ
    #    by a wake-up or two: after each wake-up the plugin starts listening afresh, which can move a later score across the threshold.
    live, predicted = len(run_stream(OVOS_HOTWORD, train_audio[FIT_END:])[0]), len(wake_ups(smoke_scores, THRESHOLD))
    margin = max(2, live // 10)
    print(f"SMOKE check: on the set-aside audio the plugin woke {live} times; predicted from its scores: {predicted} (must be within {margin})")
    assert abs(live - predicted) <= margin, "the wake-ups predicted from recorded scores differ from the plugin's"
    # 5. separation that does not depend on the threshold: the chance that a wake-word block scores above another block
    blk = row_label[val_rows]
    other = np.sort(smoke_scores[blk == 0])
    auc = float((np.searchsorted(other, smoke_scores[blk == 1], "left") + np.searchsorted(other, smoke_scores[blk == 1], "right")).sum()
                / (2 * (blk == 1).sum() * len(other)))
    print(f"SMOKE check: a wake-word block scores above another block with probability {auc:.3f} (must exceed 0.9)")
    assert auc > 0.9, "the saved model, run by the plugin, does not separate the phrase from the rest"
    # 6. at the shipped settings the plugin wakes on at least a quarter of the test clips (a tiny robot-voice model, so the bar is low)
    print(f"SMOKE check: {CTX_DET} of {CTX_N} test clips detected at the shipped settings (must be at least {max(1, CTX_N // 4)})")
    assert CTX_DET >= max(1, CTX_N // 4), "the saved model, at its own threshold, misses nearly all of the phrase"

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.